In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input
from tensorflow.keras.callbacks import Callback, EarlyStopping


## Simulation1b

[Id --->> orth]


In [ ]:
X_id = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)
Y = pd.read_excel('orthograph_sim1_2.xlsx').values[0:477,0:350].astype(np.float32)


in_id   = Input(shape=(35,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)

combined = tower_id
h = Dense(500, activation="sigmoid")(combined)
output = Dense(350, activation="sigmoid")(h)

model1d = Model(inputs=[in_id], outputs=output)


In [ ]:
model1d.compile(optimizer="adam", loss="mse")
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)

model1d.fit(X_id,Y,batch_size=25, epochs=700, callbacks=[callback])
model1d.save('simulation_1d_model.h5')

Epoch 1/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1203
Epoch 2/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0182
Epoch 3/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0178
Epoch 4/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0179
Epoch 5/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0177
Epoch 6/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0177
Epoch 7/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0177
Epoch 8/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0178
Epoch 9/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0181
Epoch 10/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0176
Epoch 11/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0175
Epoch 12/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0175
Epoch 13/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0177
Epoch 14/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0175
Epoch 15/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - los

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
Y = pd.read_excel('orthograph_sim1_2.xlsx').values[0:477,0:350].astype(np.float32)



data_lst= ["TL+Allog1", "TL-Allog1", "SubL+Allog1", "SubL-Allog1"]
df_sim1d = pd.DataFrame()
for document in data_lst:
    X_id   = pd.read_excel(document+'.xlsx', sheet_name='identity', index_col=0).values.astype(np.float32)
    #X_pos  = pd.read_excel(document+'.xlsx', sheet_name='position', index_col=0).values.astype(np.float32)

    pred = model1d.predict(X_id)
    valz = []
    for i in range(len(Y)):
      r = np.corrcoef(pred[i], Y[i])[0,1]
      valz.append(round(1-r,3))


    df_sim1d[f'target-{document}']= valz

df_sim1d.to_excel('simulation_1d_correlations_test_5.xlsx')

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


# Correlation Distance Between words that share the same root

In [ ]:
import itertools

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model1d.predict(X_id)

df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


# Load roots file
df_roots = pd.read_excel('targets with roots and semantics.xlsx', sheet_name='grouped by roots',     index_col=0)


# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')


# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('Root'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(350)]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'Root': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('Root')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()


with pd.ExcelWriter('simulation1d_root_pairwise_distances_5.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_root_summary', index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


# Correlation Distance Between the word and other 4 conditions

In [ ]:
df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model1d.predict(X_id)


df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


PAIRS_XLSX  = 'targets with roots and semantics.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)


out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation1d_4_conditions_pairs_with_distance_5.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
